In [ ]:
import os

os.environ.setdefault("XLA_FLAGS", "--xla_force_host_platform_device_count=15")
# os.environ["NUMBA_DISABLE_JIT"] = "1"
# os.environ["JAX_COMPILATION_CACHE_DIR"] = os.path.expanduser("~/.cache/jax_euclid")

import jax

jax.config.update("jax_enable_x64", True)
jax.config.update("jax_platform_name", "cpu")
jax.config.update("jax_enable_compilation_cache", True)
jax.config.update("jax_persistent_cache_min_compile_time_secs", 1.0)
print("JAX devices:", jax.devices())

In [ ]:
import matplotlib.pyplot as plt

from gwemfish.simple_pipeline import (
    _deep_merge_dict,
    make_default_cfg,
    plot_system_observation,
    setup_em_observation,
    setup_gw_observation,
    run_inference, 
    plot_posterior, 
    to_source_plane_samples, 
    plot_source_posterior,
)
#
#OUTPUT_DIR = os.path.join("examples", "outputs", "simple_pipeline_demonstration")
#os.makedirs(OUTPUT_DIR, exist_ok=True)
#print("OUTPUT_DIR:", os.path.abspath(OUTPUT_DIR))

In [ ]:
from gwemfish.config import DEFAULT_KWARGS_NUMERICS, SOLVER_PARAMS
OUTPUT_DIR = "figures"
os.makedirs(OUTPUT_DIR, exist_ok=True)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
import scienceplots
plt.style.use(["science", "ieee", "high-vis"])
plt.rcParams["text.usetex"] = False


In [ ]:
from lenstronomy.SimulationAPI.ObservationConfig.Euclid import Euclid
from lenstronomy.SimulationAPI.observation_api import SingleBand

_euclid_inst = Euclid("VIS", "GAUSSIAN")
_euclid_cfg  = _euclid_inst.kwargs_single_band()
_euclid_band = SingleBand(**_euclid_cfg)

EUCLID_PIX_SCL  = _euclid_cfg["pixel_scale"]                                          # 0.101 arcsec
EUCLID_FWHM     = _euclid_cfg["seeing"]                                               # 0.16 arcsec
EUCLID_T_EXP    = _euclid_cfg["exposure_time"] * _euclid_cfg.get("num_exposures", 1)  # 2264 s total
EUCLID_BKG_RMS  = _euclid_band.background_noise                                       # ~0.0109 e/s/px
EUCLID_MAG_ZP   = _euclid_cfg["magnitude_zero_point"]                                 # 25.72
NPIX            = 80                                                                  # 60 px ≈ 6 arcsec FOV
print(EUCLID_PIX_SCL)
print(EUCLID_FWHM)
print(EUCLID_T_EXP)
print(EUCLID_BKG_RMS)
print(EUCLID_MAG_ZP)
print(NPIX)

sample_cfg = make_default_cfg()
sample_cfg["em"]["pixel_grid_kwargs"]           = {"npix": NPIX, "pix_scl": EUCLID_PIX_SCL}
sample_cfg["em"]["psf_kwargs"]                  = {"psf_type": "GAUSSIAN", "fwhm": EUCLID_FWHM}
sample_cfg["em"]["noise_simu_kwargs"]           = {"npix": NPIX, "background_rms": EUCLID_BKG_RMS, "exposure_time": EUCLID_T_EXP}
sample_cfg["em"]["noise_inf_kwargs"]            = {"npix": NPIX, "background_rms": None, "exposure_time": EUCLID_T_EXP}
sample_cfg["em"]["exposure_time"]               = EUCLID_T_EXP
sample_cfg["em"]["seed"]                        = 87651
sample_cfg["gw"]["image_box_half_width"]        = 5.0
sample_cfg["gw"]["error_scales"]["sigma_dL_eff"]= 0.3
sample_cfg["gw"]["error_scales"]["sigma_td"]    = 0.001
sample_cfg["use_parameter_layout"]              = True
sample_cfg["output"]["output_dir"]              = OUTPUT_DIR

In [ ]:
sample_cfg["gw"]["solver_params"]["search_window"] = 2

In [ ]:
# sample_cfg['gw']

In [ ]:
from lenstronomy.Util import param_util
from lenstronomy.SimulationAPI.mag_amp_conversion import MagAmpConversion

In [ ]:
import copy

def row_to_cfg(row, sample_cfg, gw_enabled):
    """
    Convert one row of the Qiuhan's sky catalog into a GWEMFISH configuration.

    Parameters
    ----------
    row : pandas.Series
        One row of the lens catalog.

    sample_cfg : dict
        Reference GWEMFISH config.

    Returns
    -------
    cfg : dict
        GWEMFISH configuration for this lens system.
    """

    cfg = copy.deepcopy(sample_cfg)

    # ==========================================================
    # Convert axis ratio + PA -> ellipticity
    # ==========================================================

    # --- 1. PREPARE LENS (DEFLECTOR) LIGHT PARAMETERS ---
    # Convert q and pa to lenstronomy ellipticity (e1, e2)
    lens_e1, lens_e2 = param_util.phi_q2_ellipticity(phi=row['deflector_pa'], q=row['deflector_q'])


    kwargs_lens_light_mag = [{
    'magnitude': row['deflector_app_mag_VIS'],
    'R_sersic': row['deflector_Re'],
    'n_sersic': 4.0,  # Standard assumption for elliptical lens galaxies
    'e1': lens_e1, 'e2': lens_e2,
    'center_x': 0, 'center_y': 0
    }]

    # --- 2. PREPARE SOURCE LIGHT PARAMETERS ---
    # Convert q and pa to lenstronomy ellipticity (e1, e2)
    source_e1, source_e2 = param_util.phi_q2_ellipticity(phi=row['source_pa'], q=row['source_q'])
    
    # Choose your band (e.g., VIS)
    kwargs_source_mag = [{
        'magnitude': row['source_app_mag_VIS'], 
        'R_sersic': row['source_Re'],
        'n_sersic': row['source_sersic_index'],
        'e1': source_e1, 'e2': source_e2,
        'center_x': row['source_relative_x'], 
        'center_y': row['source_relative_y']
    }]
    
    # --- 3. CONVERT BOTH TO AMP ---
    # Define your model profile types
    kwargs_model = {
        'lens_light_model_list': ['SERSIC_ELLIPSE'],
        'source_light_model_list': ['SERSIC_ELLIPSE']
    }
    
    # Initialize the converter with your survey zero-point
    mag_converter = MagAmpConversion(kwargs_model=kwargs_model, magnitude_zero_point=25.9)
    
    # Get the final dictionaries containing the calculated 'amp' keys
    lens_light_amp = mag_converter.magnitude2amplitude(kwargs_lens_light_mag=kwargs_lens_light_mag)
    source_amp = mag_converter.magnitude2amplitude(kwargs_source_mag=kwargs_source_mag)
    
    #print(lens_light_amp)
    #print(source_amp)
    
    #source_e1, source_e2 = q_pa_to_e1e2(
     #   row["source_q"],
      #  row["source_pa"],
   # )

    lens_pos = (float(row['deflector_ra']), float(row['deflector_dec']))
    
    source_pos = (float(row["source_relative_x"]), float(row["source_relative_y"]))

    # ==========================================================
    # Lens geometry
    # ==========================================================

    cfg["lens"]["zl"] = float(row["deflector_z"])
    cfg["lens"]["zs"] = float(row["source_z"])

    # ==========================================================
    # Lens mass model (EPL)
    # ==========================================================

    cfg["lens"]["kwargs_lens"][0]["theta_E"] = float(row["deflector_thetaE"])
    cfg["lens"]["kwargs_lens"][0]["gamma"] = float(row["deflector_slope"])

    cfg["lens"]["kwargs_lens"][0]["e1"] = float(lens_e1)
    cfg["lens"]["kwargs_lens"][0]["e2"] = float(lens_e2)

    cfg["lens"]["kwargs_lens"][0]["center_x"] = 0.00
    cfg["lens"]["kwargs_lens"][0]["center_y"] = 0.00

    # ==========================================================
    # External shear
    # ==========================================================

    cfg["lens"]["kwargs_lens"][1]["gamma1"] = float(row["deflector_shear1"])
    cfg["lens"]["kwargs_lens"][1]["gamma2"] = float(row["deflector_shear2"])

    cfg["lens"]["kwargs_lens"][1]["ra_0"] = 0.00
    cfg["lens"]["kwargs_lens"][1]["dec_0"] = 0.00

    # ==========================================================
    # Source light
    # ==========================================================

    cfg["em"]["kwargs_source"][0]["R_sersic"] = float(row["source_Re"])
    cfg["em"]["kwargs_source"][0]["n_sersic"] = float(row["source_sersic_index"])

    cfg["em"]["kwargs_source"][0]["e1"] = float(source_e1)
    cfg["em"]["kwargs_source"][0]["e2"] = float(source_e2)

    cfg["em"]["kwargs_source"][0]["center_x"] = source_pos[0]
    cfg["em"]["kwargs_source"][0]["center_y"] = source_pos[1]

    cfg["em"]["kwargs_source"][0]["amp"] = float(source_amp[1][0]['amp'])

    # ==========================================================
    # Lens light
    # ==========================================================

    cfg["em"]["kwargs_lens_light"][0]["R_sersic"] = float(row["deflector_Re"])
    cfg["em"]["kwargs_lens_light"][0]["n_sersic"] = float(4)

    cfg["em"]["kwargs_lens_light"][0]["e1"] = float(lens_e1)
    cfg["em"]["kwargs_lens_light"][0]["e2"] = float(lens_e2)

    cfg["em"]["kwargs_lens_light"][0]["center_x"] = 0.00
    cfg["em"]["kwargs_lens_light"][0]["center_y"] = 0.00

    cfg["em"]["kwargs_lens_light"][0]["amp"] = float(lens_light_amp[0][0]['amp'])

    # ==========================================================
    # GW source position
    # ==========================================================
    cfg["em"]["source_pos"] = source_pos
    
    if gw_enabled is True:
        cfg["gw"]["source_pos"] = (source_pos[0]+0.005, source_pos[1]-0.005)#(0.0001,0.0002)##(0.01,0.02)#

    if gw_enabled is False:
        cfg["gw"] = {"enabled": False}

    return cfg

## Prepare to implement data

In [ ]:
import pandas as pd

In [ ]:
df = pd.read_csv('../catalog/filtered_lens_catalog_PL_IC_gt_70.csv')

In [ ]:
df.columns

ID= 39<br>
GW-only: Problem in helens solver, it was thrwoing a valid image via remove central image (FIXED that in notebook and worked, not implemented in GWEMFISH yet) <br>
EM-only: low SNR, making source_amp*100 worked <br>
EM+GW: worked (keep in mind lens1_ra_0 and lens1_dec_0 must be fixed)<br>

ID= 1478<br>
EM data is not measuring lens_gamma for this case
GW-only:  Need central image removing at truth<br>
EM-only:  Sersic index of source (~0.6) is outside the default prior range [0.8,5] that is why sampler did not sample and Nan at r-hat values, after fixing that now still Nan in param sigmas, inflating source amp worked indicating low SNR (fixing gamma and lens light params worked)<br>
EM+GW: fine (fix lens light + lens gamma)<br>

ID= 917<br>
source is too dim ~4 , lens~37 <br>
GW-only:  Need central image removing at truth<br>
EM-only:  increasing source_amp by 5 times worked (no need to fix lens light also)<br>
EM+GW: <br>

ID= 1529<br>
source ~4 and lens 115 <br>
GW-only:  Need central image removing at truth<br>
EM-only:  increasing source_amp by 3 times worked (no need to fix lens light also)<br>
EM+GW: <br>

ID= 2417<br>
source ~8 lens ~11  <br>
GW-only:  Need central image removing at truth<br>
EM-only:  increasing source_amp by 3 times worked (no need to fix lens light also)<br>
EM+GW: <br>

ID= 790<br>
source ~7 lens ~17  <br>
GW-only:  Need central image removing at truth<br>
EM-only:  increasing source_amp by 5/3/10 times worked (no need to fix lens light also)<br>
EM+GW: <br>

ID= 1035<br>
source ~2 lens ~1  <br>
GW-only:  Need central image removing at truth<br>
EM-only:  need to increase snr, also lens is too dim we need to fix lens light otherwise lens light params will be degenerate<br>
EM+GW: <br>

ID= 425<br>
source ~5 lens ~13 ,larger theta_E than all previous cases (lensed arcs are far from lens center)<br>
GW-only:  Need central image removing at truth<br>
EM-only:  works<br>
EM+GW: <br>


### Pick a galaxy that will be the correct source and lens pair

### failing instances for EM modes (but GW-only mode is good)
- 1478 (all looks good and pretty, but PE fails)
- 917
- 1529
- 2417
- 790
- 1035 (looks very dim source and lens)
- 425 (EM inference also looks just fine, BUT too faint)


### Failing instance for GW only mode
- 1417 - working with normal settings, not an issue of GWEMFISH
- 857  - working with normal settings, Tstar and DL precision low, if we reduce DL measureement error will get good posterior , not an issue of GWEMFISH
- 1122 - Fold image configuration, helens misses one of the two images near critical curve, even after fixing that (use jaxtronomy as a patch in the notebook, not yet in GWEMFISH) highly degenerate, geting potreriors now
- 749  - 3 image system, making it 4 works
- 1064 - works 

In [ ]:
id_test = 1064#749#1122#857#1417#425#1035#1529#2417#917#1478 #39
source_galaxy = df.iloc[id_test:id_test+1]

In [ ]:
row = source_galaxy.iloc[0]
cfg_gw_source = row_to_cfg(row, sample_cfg, True)
# cfg_gw_source["use_parameter_layout"] = True

In [ ]:
print(cfg_gw_source['em']['kwargs_source'])

In [ ]:
print(cfg_gw_source['em']['kwargs_lens_light'])

cfg_gw_source['gw']['source_pos'] = (cfg_gw_source['gw']['source_pos'][0]-0.005, cfg_gw_source['gw']['source_pos'][1]+0)

In [ ]:
# cfg_gw_source

In [ ]:
cfg_gw_source['gw']

In [ ]:
ctx_gw_source = setup_em_observation(cfg=cfg_gw_source)

In [ ]:
ctx_gw_source = setup_gw_observation(ctx_gw_source, cfg=cfg_gw_source)
# ctx_gw_source = setup_gw_observation({}, cfg=cfg_gw_source)

In [ ]:
ctx_gw_source

In [ ]:
from gwemfish import prune_gw_images
if len(ctx_gw_source["x_img_gw"]) > 4:
    ctx_gw_source = prune_gw_images(ctx_gw_source, n_keep=4)   # also sets ctx["cfg"]["gw"]["n_images"] = 4

In [ ]:
tp = ctx_gw_source['truth_params']

In [ ]:
# tp

In [ ]:
fig = plot_system_observation(
    ctx_gw_source,
    cfg={"output": {"save_system_plot_path": None}} #os.path.join(OUTPUT_DIR, "system_observation.png")}},
)
plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from gwemfish import compute_noise_snr_maps


data = np.asarray(ctx_gw_source["em_obs"]["data"])
noise_map, snr_map = compute_noise_snr_maps(ctx_gw_source)
bg_rms = float(ctx_gw_source["cfg"]["em"]["noise_simu_kwargs"]["background_rms"])
data_log = np.log10(np.clip(data, bg_rms, None))

xx, yy = ctx_gw_source["pixel_grid"].pixel_coordinates
x_img = np.asarray(ctx_gw_source["x_img_gw"]).ravel()
y_img = np.asarray(ctx_gw_source["y_img_gw"]).ravel()

fig, (ax_log, ax_snr) = plt.subplots(1, 2, figsize=(10, 4))
im0 = ax_log.pcolormesh(xx, yy, data_log, shading="auto")
im1 = ax_snr.pcolormesh(xx, yy, snr_map, shading="auto")
fig.colorbar(im0, ax=ax_log, label=r"$\log_{10}$ noisy $e^-$/s (floor = bg rms)")
fig.colorbar(im1, ax=ax_snr, label="S/N")
ax_log.set_title("Noisy EM observation (log)")
ax_snr.set_title("S/N map")
for ax in (ax_log, ax_snr):
    ax.scatter(x_img, y_img, s=40, facecolors="none", edgecolors="red")
    ax.set_xlabel("RA [arcsec]")
    ax.set_ylabel("Dec [arcsec]")
    ax.set_aspect("equal")
plt.tight_layout()
plt.show()

print(f"bg_rms = {bg_rms:.4g}  peak data = {data.max():.4g}  peak S/N = {snr_map.max():.2f}")
for i, (x, y) in enumerate(zip(x_img, y_img)):
    ix = np.unravel_index(np.argmin((xx - x)**2 + (yy - y)**2), xx.shape)
    print(f"  image {i+1}: S/N = {float(snr_map[ix]):.2f}  data = {float(data[ix]):.4g}")


In [ ]:
#Turn this on
# helens' triangle-subdivision search misses one of the 4 true images for
# this system (id_test=1122) regardless of nsolutions/niter/nsubdivisions/
# scale_factor (swept and confirmed) -- it returns a spurious (0,0) padding
# slot instead of the real 4th image. jaxtronomy's closed-form EPL(+SHEAR)
# 'analytical' solver recovers all 4 true images exactly (max|dr| ~ 2e-11)
# for this system, at every magnification_limit tested. This cell swaps
# only the coarse-localize step; the Newton-polish + custom_root (implicit
# function theorem) differentiation is unchanged and solver-agnostic -- see
# gwemfish/differentiable_solver.py.
#
# Notebook-local override: monkeypatches gwemfish.lens_setup's module-level
# setup_differentiable_helens_solver (picked up by
# _build_inference_probmodel_source_plane's local import), no gwemfish core
# changes.

import jax
import jax.numpy as jnp
import numpy as np

import gwemfish.config as gwcfg
import gwemfish.lens_setup as _lens_setup
from gwemfish.differentiable_solver import polish_image
from jaxtronomy.LensModel.lens_model import LensModel
from jaxtronomy.LensModel.Solver.lens_equation_solver import LensEquationSolver as JaxLensEquationSolver


class DifferentiableLensEquationSolverJaxtronomy:
    """Drop-in replacement for DifferentiableLensEquationSolver: same
    .solve(beta, kwargs_lens, nsolutions=..., **_ignored) signature, (theta, beta)
    return shape (N, 2) each. Coarse localize via jaxtronomy's closed-form
    EPL(+SHEAR) 'analytical' solver (host callback -- it returns a
    variable-length result) instead of helens' triangle search; same
    stop-gradient + Newton-polish + custom_root pattern otherwise.
    """

    def __init__(self, lens_model_list, z_lens, z_source, ray_shooting_func,
                 n_newton=8, magnification_limit=1e-3):
        self._lensModel = LensModel(lens_model_list=lens_model_list, z_lens=z_lens, z_source=z_source)
        self._jax_solver = JaxLensEquationSolver(self._lensModel)
        self._ray_shooting_func = ray_shooting_func
        self._n_newton = n_newton
        self._magnification_limit = magnification_limit
        self._polish_batched = jax.vmap(self._polish_one, in_axes=(0, None, None))

    def _polish_one(self, theta_guess, beta, kwargs_lens):
        return polish_image(theta_guess, beta, kwargs_lens, self._ray_shooting_func,
                             n_newton=self._n_newton)

    def _host_localize(self, bx, by, kwargs_lens_np, nsolutions):
        kwargs_lens = [{k: float(v) for k, v in kw.items()} for kw in kwargs_lens_np]
        x_img, y_img = self._jax_solver.image_position_from_source(
            float(bx), float(by), kwargs_lens,
            solver="analytical", magnification_limit=self._magnification_limit,
        )
        x_img = np.asarray(x_img, dtype=np.float64).ravel()
        y_img = np.asarray(y_img, dtype=np.float64).ravel()
        n_found = min(len(x_img), nsolutions)
        theta0 = np.zeros((nsolutions, 2), dtype=np.float64)
        theta0[:n_found, 0] = x_img[:n_found]
        theta0[:n_found, 1] = y_img[:n_found]
        return theta0

    def solve(self, beta, kwargs_lens, nsolutions=5, **_ignored_helens_kwargs):
        beta_sg = jax.lax.stop_gradient(beta)
        kwargs_lens_sg = jax.tree_util.tree_map(jax.lax.stop_gradient, kwargs_lens)

        result_shape = jax.ShapeDtypeStruct((nsolutions, 2), jnp.float64)
        theta0_all = jax.pure_callback(
            lambda bx, by, kl: self._host_localize(bx, by, kl, nsolutions),
            result_shape, beta_sg[0], beta_sg[1], kwargs_lens_sg,
            vmap_method="sequential",
        )
        theta0_all = jax.lax.stop_gradient(theta0_all)
        is_padding_slot = jnp.hypot(theta0_all[:, 0], theta0_all[:, 1]) < 1e-8
        # Newton-polishing the exact (0,0) padding slot hits the deflection
        # Jacobian's genuine singularity at the lens center -> NaN. jnp.where
        # below picks the *value* from the untouched branch, but its gradient
        # still differentiates both branches (0 * NaN = NaN), so nudge the
        # padding slot's *input* off the singularity before polishing; the
        # true (0,0) marker (theta0_all, unperturbed) is still what gets
        # selected in the output.
        theta0_for_polish = jnp.where(is_padding_slot[:, None], theta0_all + 1e-6, theta0_all)
        theta_polished = self._polish_batched(theta0_for_polish, beta, kwargs_lens)

        theta_final = jnp.where(is_padding_slot[:, None], theta0_all, theta_polished)
        beta_final = jnp.broadcast_to(beta, (nsolutions, 2))
        return theta_final, beta_final


def setup_differentiable_jaxtronomy_solver(pixel_grid, lens_gw, pixel_scale_factor=0.8,
                                            solver_params=None, n_newton=8):
    if solver_params is None:
        solver_params = gwcfg.SOLVER_PARAMS.copy()
    lens_model_list = cfg_gw_source["lens"]["lens_model_list"]
    zl = cfg_gw_source["lens"]["zl"]
    zs = cfg_gw_source["lens"]["zs"]
    solver = DifferentiableLensEquationSolverJaxtronomy(
        lens_model_list, zl, zs, lens_gw.ray_shoot,
        n_newton=n_newton, magnification_limit=1e-3,
    )
    return solver, pixel_grid, solver_params


_lens_setup.setup_differentiable_helens_solver = setup_differentiable_jaxtronomy_solver

print("Patched: source-plane differentiable solver now uses jaxtronomy 'analytical' "
      "(EPL+SHEAR closed-form) as coarse localizer instead of helens' triangle search.")


In [ ]:
import gwemfish.config as gwcfg
print("Current SOLVER_PARAMS:", gwcfg.SOLVER_PARAMS)

# # Increase grid resolution so all 4 images are found uniquely
# gwcfg.SOLVER_PARAMS["nsubdivisions"] = 2

In [ ]:
# ctx_gw_source

In [ ]:
# import autolens.plot as aplt
# from gwemfish import simulate_in_pal, plot_system_observation_pal

# ctx_pal = simulate_in_pal(ctx_gw_source)

# # tracer plot
# plot_system_observation_pal(
#     ctx_pal,
#     cfg={
#         "plot":   {"pal_plot_dataset": True, "pal_plot_tracer": True},
#         "output": {"save_pal_tracer_plot_path": None, "save_pal_dataset_plot_path": None},
#     }
# )

# # gwemfish dataset only (data / noise / PSF / S-N)
# # aplt.subplot_imaging_dataset(ctx_pal["dataset_gwemfish"])

In [ ]:
# from gwemfish.lens_setup import setup_differentiable_helens_solver
# from gwemfish.lens_setup import remove_central_image
# import jax.numpy as jnp

# solver, _, solver_params = setup_differentiable_helens_solver(
#     ctx_gw_source_gw["pixel_grid"], ctx_gw_source_gw["lens_gw"]
# )
# src = jnp.array([ctx_gw_source_gw["cfg"]["gw"]["source_pos"][0],
#                  ctx_gw_source_gw["cfg"]["gw"]["source_pos"][1]])
# thetas, _ = solver.solve(src, ctx_gw_source_gw["kwargs_lens"], **solver_params)
# tx, ty, _, _ = remove_central_image(thetas, _, 0.0, 0.0)

# print("helens  x_img:", jnp.sort(tx))
# print("ctx     x_img:", jnp.sort(jnp.array(ctx_gw_source_gw["x_img_gw"])))
# print("diff:", jnp.sort(tx) - jnp.sort(jnp.array(ctx_gw_source_gw["x_img_gw"])))

## Set priors

In [ ]:
import numpyro.distributions as dist

In [ ]:
import copy

ctx_gw_source_em = copy.deepcopy(ctx_gw_source)
ctx_gw_source_gw = copy.deepcopy(ctx_gw_source)
ctx_gw_source_both = copy.deepcopy(ctx_gw_source)

In [ ]:
ctx_gw_source_gw["cfg"]["priors"] = {
        #"T_star": float(tp["T_star"]),
        #"dL": float(tp["dL"]),
        'lens0_gamma': float(tp['lens0_gamma']),#dist.Uniform(0.001, 3.0),#dist.TruncatedNormal(tp['lens0_gamma'], 0.1, low=0.001, high=5.0),
        'lens0_theta_E': float(tp['lens0_theta_E']),#dist.TruncatedNormal(tp['lens0_theta_E'], 0.1, low=0.001, high=5.0),
         #'lens0_e1': float(tp["lens0_e1"]), 
         #'lens0_e2': float(tp['lens0_e2']),
         'lens0_e1': float(tp['lens0_e1']), #dist.Normal(tp['lens0_e1'], 0.01),
         'lens0_e2': dist.Uniform(-0.9, 0.9),#dist.Normal(tp['lens0_e2'], 0.01),
         'lens0_center_x': 0.0, #dist.Normal(0, lens_prior_sigma['lens0_center_x']),
         'lens0_center_y': 0.0, #dist.Normal(0, lens_prior_sigma['lens0_center_y']),
         #'lens1_gamma1': float(tp["lens1_gamma1"]),
         #'lens1_gamma2': float(tp["lens1_gamma2"]),
         'lens1_gamma1': float(tp['lens1_gamma1']),#dist.Normal(tp['lens1_gamma1'], 0.01),
         'lens1_gamma2': float(tp['lens1_gamma2']),#dist.Normal(tp['lens1_gamma2'], 0.01),
         'lens1_ra_0': 0.0,
         'lens1_dec_0': 0.0,
        #   'light0_R_sersic': float(tp["light0_R_sersic"]),
        #   'light0_n_sersic': float(tp["light0_n_sersic"]),
        #   'light0_amp': float(tp['light0_amp']),
        #  'light0_e1': float(tp["light0_e1"]),
        #  'light0_e2': float(tp["light0_e2"]),
        #  'light0_center_x': float(tp["light0_center_x"]),
        #  'light0_center_y': float(tp["light0_center_y"]),
        #   "noise_sigma_bkg": tp["noise_sigma_bkg"],  
        } 

In [ ]:
# import jax.numpy as jnp
# import helens.solver as _helens_solver

# def _subdivide_triangles_fixed(self, triangles, niter=1):
#     """NumPy-2.x / JAX compat patch: use jnp.stack instead of bare Python lists."""
#     v1, v2, v3 = triangles.transpose(1, 0, 2)
#     v4 = 0.5 * (v1 + v2)
#     v5 = 0.5 * (v2 + v3)
#     v6 = 0.5 * (v3 + v1)
#     t1 = jnp.stack([v1, v4, v6])
#     t2 = jnp.stack([v4, v2, v5])
#     t3 = jnp.stack([v6, v4, v5])
#     t4 = jnp.stack([v6, v5, v3])
#     subtriangles = jnp.column_stack((t1, t2, t3, t4)).transpose(1, 0, 2)
#     for k in range(1, niter):
#         v1, v2, v3 = subtriangles.transpose(1, 0, 2)
#         v4 = 0.5 * (v1 + v2)
#         v5 = 0.5 * (v2 + v3)
#         v6 = 0.5 * (v3 + v1)
#         t1 = jnp.stack([v1, v4, v6])
#         t2 = jnp.stack([v4, v2, v5])
#         t3 = jnp.stack([v6, v4, v5])
#         t4 = jnp.stack([v6, v5, v3])
#         subtriangles = jnp.column_stack((t1, t2, t3, t4)).transpose(1, 0, 2)
#     return subtriangles.reshape(4**niter * len(triangles), 3, 2)

# _helens_solver.LensEquationSolver._subdivide_triangles = _subdivide_triangles_fixed

In [ ]:
import jax.numpy as jnp
import numpy as np
from gwemfish.data_sim import compute_gw_from_images
from gwemfish.lens_setup import remove_central_image

# fisher-source is instant (no NUTS); it populates ctx["likelihood"]["probmodel"]
run_inference(ctx_gw_source_gw, mode="GW-only", method="fisher-source",
              cfg={"output": {"json_tag": "fisher_source_preflight"}})

pm  = ctx_gw_source_gw["likelihood"]["probmodel"]
tp  = ctx_gw_source_gw["truth_params"]
kl  = ctx_gw_source_gw["kwargs_lens"]
src = ctx_gw_source_gw["cfg"]["gw"]["source_pos"]

thetas, betas_out = pm.solver.solve(jnp.array([float(src[0]), float(src[1])]), kl, **pm.solver_params)
tx_nc, ty_nc, _, _ = remove_central_image(thetas, betas_out, 0.0, 0.0)
x_m, y_m = np.array(tx_nc), np.array(ty_nc)

print(f"solver_params : {pm.solver_params}")
print(f"\nModel images (nsolutions={pm.solver_params['nsolutions']}):")
for x, y in zip(x_m, y_m):
    print(f"  x={x:.6f}  y={y:.6f}  r={np.hypot(x, y):.6f}")
print("Mock images:")
for x, y in zip(ctx_gw_source_gw["x_img_gw"], ctx_gw_source_gw["y_img_gw"]):
    print(f"  x={float(x):.6f}  y={float(y):.6f}  r={np.hypot(float(x), float(y)):.6f}")

r_m    = np.sort(np.hypot(x_m, y_m))
r_mock = np.sort([np.hypot(float(x), float(y)) for x, y in zip(ctx_gw_source_gw["x_img_gw"], ctx_gw_source_gw["y_img_gw"])])
print(f"\nMax |Δr|  = {np.max(np.abs(r_m - r_mock)) if len(r_m)==len(r_mock) else float('nan'):.2e}  (should be < 1e-6)")

_, td_m, *_ = compute_gw_from_images(jnp.array(x_m), jnp.array(y_m), kl, pm.lens_gw,
                                      float(tp["T_star"]), float(tp["dL"]))
td_m   = np.sort(np.array(td_m))
td_obs = np.sort(np.array(ctx_gw_source_gw["gw_obs"]["time_delays"]))
print(f"Max |Δtd| = {np.max(np.abs(td_m - td_obs)):.2e} s  (should be < 0.01 s)")


In [ ]:
# ctx_gw_source_gw

In [ ]:
samples, truths = run_inference(ctx_gw_source_gw, mode="GW-only", method="fisher-source")

H0 = ctx_gw_source_gw['fisher']['H0']
g0 = ctx_gw_source_gw['fisher']['g0']

import jax.numpy as jnp
print(jnp.isnan(H0).any(), jnp.isinf(H0).any())
print(jnp.isnan(g0).any())
print(jnp.diag(H0))          # per-param curvature, compare to the printed diagnostics


In [ ]:
FM = -H0   # (T_star/dL block ~1e-8, rest ~1e9-1e10)
cov = jnp.linalg.inv(FM)
print(jnp.isnan(cov).any())        # False — no NaN
print(jnp.diag(cov))               # finite, even reasonable-looking numbers
print(jnp.linalg.eigvalsh(cov))    # [-1.2e-09, -2.9e-10, -9.9e-11, +3.4e+07, +7.4e+07]
                                    #  ^^^ 3 negative eigenvalues — cov isn't PD
key = jax.random.PRNGKey(0)
u0 = jnp.array(ctx_gw_source_gw["likelihood"]["u0"])
samp = jax.random.multivariate_normal(key, u0, cov, shape=(5,))  # default method='cholesky'
print(jnp.isnan(samp).any())       # True — full NaN


In [ ]:
print(g0)
print(g0 / jnp.sqrt(jnp.abs(jnp.diag(H0))))   # rough offset-from-mode, in sigma units


In [ ]:
# # after a fisher-source / this Hessian print
# # T_star σ ≈ 2.4e4, dL σ ≈ 6e3, y σ ≈ 7e-5
# # tp = ctx_gw_source_gw["truth_params"]
# ctx_gw_source_gw["cfg"]["priors"]["T_star"] = dist.Uniform(
#     float(tp["T_star"]) - 20 * 2.4e4, float(tp["T_star"]) + 20 * 2.4e4
# )
# ctx_gw_source_gw["cfg"]["priors"]["dL"] = dist.Uniform(
#     float(tp["dL"]) - 20 * 6.1e3, float(tp["dL"]) + 20 * 6.1e3
# )
# ctx_gw_source_gw["cfg"]["gw"]["source_box_half_width"] = 0.005  # optional; 0.05 is not the root cause

In [ ]:
samples_deriv_approx_source, truths_deriv_approx_source = run_inference(
    ctx_gw_source_gw,
    mode="GW-only",
    method="deriv-approx-source",#"fisher-source",#"deriv-approx-source",
    cfg={
        "output": {"json_tag": "deriv_approx_source"},
        "inference": {
            "informed":   True,
            "regularize": False,
            "num_chains":  8,
            "num_warmup":  8000,   # smoke: 200; production: 9500
            "num_samples": 12000,   # smoke: 300; production: 14500
        },
    },
)

In [ ]:
# samples_deriv_approx_source

In [ ]:
import corner

_ = plot_posterior(
        samples_deriv_approx_source,
        truths_deriv_approx_source,
        cfg={
            "output": {"output_dir": None},
            "plot": {
                "plot_mode": "combined",
                "save_path": None,
            },
        },
    )

In [ ]:
# import os
# import numpy as np

# NAUTILUS_SIGMA_SPAN      = 2.0
# NAUTILUS_CHECKPOINT      = "outputs/nautilus_source_fix_gamma_checkpoint.hdf5"
# # NAUTILUS_CHECKPOINT    = "outputs/nautilus_source_fix_Tstar_Dl_checkpoint.hdf5"
# NAUTILUS_RESUME          = False  # True → resume existing checkpoint; False → fresh run
# NAUTILUS_N_LIVE          = 500    # smoke: 100; production: 500+
# NAUTILUS_N_EFF           = 5000    # smoke: 500; production: 10000
# NAUTILUS_DISCARD_EXPLOR  = True  # discard exploration-phase samples from posterior

# os.makedirs(os.path.dirname(NAUTILUS_CHECKPOINT), exist_ok=True)

# # ##Trun it on if we need priors from fisher-source
# # # fisher-source is the precursor: populates ctx["fisher"] + ctx["likelihood"]
# # if "fisher" not in ctx_gw_source_gw:
# #     run_inference(ctx_gw_source_gw, mode="GW-only", method="fisher-source",
# #                   cfg={"output": {"json_tag": "fisher_source_preflight"}})

# # keys   = ctx_gw_source_gw["likelihood"]["keys_to_include"]
# # u0     = np.asarray(ctx_gw_source_gw["likelihood"]["u0"])
# # H0     = np.asarray(ctx_gw_source_gw["fisher"]["H0"])
# # FM     = -H0
# # try:
# #     cov = np.linalg.inv(FM)
# # except np.linalg.LinAlgError:
# #     cov = np.linalg.pinv(FM)
# # sigmas = np.sqrt(np.diag(cov))

# # print(f"--- Nautilus priors (fisher-source H0, span={NAUTILUS_SIGMA_SPAN}σ) ---")
# # for i, key in enumerate(keys):
# #     sig = float(sigmas[i])
# #     if not np.isfinite(sig) or sig <= 0:
# #         print(f"  {key}: skip (sigma={sig}) — keeping existing prior")
# #         continue
# #     mu = float(u0[i])
# #     lo, hi = mu - NAUTILUS_SIGMA_SPAN * sig, mu + NAUTILUS_SIGMA_SPAN * sig
# #     ctx_gw_source_gw["cfg"]["priors"][key] = dist.Uniform(lo, hi)
# #     print(f"  {key}: Uniform({lo:.4g}, {hi:.4g})  [mu={mu:.4g}, σ={sig:.4g}]")

# # --- Run nautilus-source ---
# samples_nautilus_source, truths_nautilus_source = run_inference(
#     ctx_gw_source_gw,
#     mode="GW-only",
#     method="nautilus-source",
#     cfg={
#         "output": {"json_tag": "nautilus_source"},
#         "nautilus": {
#             "filepath":           NAUTILUS_CHECKPOINT,
#             "resume":             NAUTILUS_RESUME,
#             "n_live":             NAUTILUS_N_LIVE,
#             "n_eff":              NAUTILUS_N_EFF,
#             "discard_exploration": NAUTILUS_DISCARD_EXPLOR,
#         },
#     },
# )

In [ ]:
# from gwemfish import plot_posterior
# from gwemfish.corner_plot_utils import create_default_param_groups

# # "combined"  → one corner with all free params together
# # "groupwise" → one corner per physical group (lens_mass, source_light, …)
# PLOT_MODE = "combined"

# _ = plot_posterior(
#     samples_nautilus_source,
#     truths=truths_nautilus_source,
#     cfg={
#         "plot": {
#             "plot_mode": PLOT_MODE,
#             "save_path": f"outputs/nautilus_source_corner_{{group_name}}.png"
#                          if PLOT_MODE == "groupwise"
#                          else "outputs/nautilus_source_corner.png",
#             "quantiles": [0.16, 0.5, 0.84],
#             "show_titles": True,
#         },
#     },
# )

In [ ]:
# from gwemfish.corner_plot_utils import create_default_param_groups, plot_multi_comparison_corner

# free_keys = list(samples_nautilus_source.keys())

# # flip COMPARISON_MODE to switch layout:
# # "all"       → one corner with all free parameters
# # "groupwise" → one corner per physical group (lens_mass, GW, …)
# # "source"    → lens mass params + GW source position (y0gw, y1gw)
# COMPARISON_MODE = "all"

# if COMPARISON_MODE == "all":
#     param_groups = {"all": free_keys}
# elif COMPARISON_MODE == "groupwise":
#     param_groups = create_default_param_groups(samples_nautilus_source)
# else:  # "source"
#     lens_keys   = [k for k in free_keys if k.startswith("lens")]
#     source_keys = [k for k in free_keys if k in ("y0gw", "y1gw")]
#     param_groups = {"lens_and_source": lens_keys + source_keys}

# NORMALIZE = True  # True → density-normalised histograms (comparable across methods)

# # y0gw/y1gw truths live in cfg["gw"]["source_pos"], not truth_params
# src = ctx_gw_source_gw["cfg"]["gw"]["source_pos"]
# flat_truths = {**truths_nautilus_source, "y0gw": float(src[0]), "y1gw": float(src[1])}

# # truths_dict must be {group_name: {param: value}} — one sub-dict per group
# truths_dict_nested = {
#     grp_name: {k: flat_truths[k] for k in grp_keys if k in flat_truths}
#     for grp_name, grp_keys in param_groups.items()
# }
# for grp_name, grp_keys in param_groups.items():
#     missing = [k for k in grp_keys if k not in flat_truths]
#     if missing:
#         print(f"WARNING: truth missing for group '{grp_name}': {missing}")

# plot_multi_comparison_corner(
#     [samples_deriv_approx_source, samples_nautilus_source],
#     param_groups,
#     labels=["deriv-approx-source", "nautilus-source"],
#     colors=["steelblue", "darkorange"],
#     truths_dict=truths_dict_nested,
#     save_path=f"outputs/comparison_{COMPARISON_MODE}_{{group_name}}.png",
#     hist_kwargs={"density": NORMALIZE},
# )

In [ ]:
tp

# EM-only reconstruction

In [ ]:
ctx_gw_source_em["cfg"]["priors"] = {
        #'lens0_gamma': float(tp['lens0_gamma']),#dist.Uniform(0.001, 5.0),#dist.TruncatedNormal(tp['lens0_gamma'], 0.1, low=0.001, high=5.0),
        # 'lens0_gamma': dist.Uniform(0.001, 5.0),#dist.TruncatedNormal(tp['lens0_gamma'], 0.1, low=0.001, high=5.0),
        # 'lens0_theta_E': float(tp['lens0_theta_E']),#dist.Uniform(0.0001, 5.0),#dist.TruncatedNormal(tp['lens0_theta_E'], 0.1, low=0.001, high=5.0),
         #'lens0_e1': float(tp["lens0_e1"]), 
         #'lens0_e2': float(tp['lens0_e2']),
         #'lens0_e1': dist.Uniform(tp['lens0_e1']-0.08, tp['lens0_e1']+0.08),#dist.Normal(tp['lens0_e1'], 0.01),
         #'lens0_e2': dist.Uniform(tp['lens0_e2']-0.08, tp['lens0_e2']+0.08),#dist.Normal(tp['lens0_e2'], 0.01),
         'lens0_center_x': 0.0, #dist.Normal(0, lens_prior_sigma['lens0_center_x']),
         'lens0_center_y': 0.0, #dist.Normal(0, lens_prior_sigma['lens0_center_y']),
         #'lens1_gamma1': float(tp["lens1_gamma1"]),
         #'lens1_gamma2': float(tp["lens1_gamma2"]),
        #  'lens1_gamma1': float(tp['lens1_gamma1']),#dist.Uniform(tp['lens1_gamma1']-0.01, tp['lens1_gamma1']+0.01),#dist.Normal(tp['lens1_gamma1'], 0.01),
        #  'lens1_gamma2': float(tp['lens1_gamma2']),#dist.Uniform(tp['lens1_gamma2']-0.01, tp['lens1_gamma2']+0.01),#dist.Normal(tp['lens1_gamma2'], 0.01),
         'lens1_ra_0': 0.0,
         'lens1_dec_0': 0.0,
        #  'light0_R_sersic': float(tp["light0_R_sersic"]),
        #  'light0_n_sersic': float(tp["light0_n_sersic"]),
        #  'light0_amp': float(tp['light0_amp']),
        #  'light0_e1': float(tp["light0_e1"]),
        #  'light0_e2': float(tp["light0_e2"]),
         'light0_center_x': float(tp["light0_center_x"]),
         'light0_center_y': float(tp["light0_center_y"]),
        #   "noise_sigma_bkg": tp["noise_sigma_bkg"],
        #   'source0_amp': float(tp['source0_amp']),
          'source0_n_sersic': dist.Uniform(tp['source0_n_sersic']-0.5, tp['source0_n_sersic']+0.5),#float(tp['source0_n_sersic']),
        #   'source0_e1': float(tp["source0_e1"]),
        #   'source0_e2': float(tp["source0_e2"]),
        #   'source0_center_x': float(tp["source0_center_x"]),
        #   'source0_center_y': float(tp["source0_center_y"]),
         #'y0gw': dist.Normal(float(tp['y0gw']), f*float(tp['y0gw'])),
         #'y1gw': dist.Normal(float(tp['y1gw']), *float(tp['y1gw']))
        } 
# cfg_gw_source = ctx_gw_source["cfg"]

In [ ]:
samples_em_deriv_approx, truths_em_deriv_approx = run_inference(
    ctx_gw_source_em,
    mode="EM-only",
    method="deriv-approx",
    cfg={
        "output": {"json_tag": "em_deriv_approx"},
        "inference": {
            "informed": True,
            "regularize": False,
            "num_chains": 10,
            "num_warmup": 8000,
            "num_samples": 8000,
        },
    },
)

In [ ]:
# "combined"  → one corner with all free params (EM-only is typically ~6–11)
# "groupwise" → one corner per physical group (lens_mass, source_light, …)
PLOT_MODE = "groupwise"

_ = plot_posterior(
    samples_em_deriv_approx,
    truths_em_deriv_approx,
    cfg={
        "output": {"output_dir": None},
        "plot": {
            "plot_mode": PLOT_MODE,
            "save_path": None,
            "quantiles": [0.16, 0.5, 0.84],
            "show_titles": True,
        },
    },
)

# EM + GW reconstruction

In [ ]:
ctx_gw_source_both["cfg"]["priors"] = {
        'T_star': float(tp["T_star"]),
        'dL': float(tp["dL"]),
        'lens0_gamma': float(tp['lens0_gamma']),#dist.Uniform(0.001, 5.0),#dist.TruncatedNormal(tp['lens0_gamma'], 0.1, low=0.001, high=5.0),
        'lens0_theta_E': dist.Uniform(0.0001, 5.0),#dist.TruncatedNormal(tp['lens0_theta_E'], 0.1, low=0.001, high=5.0),
        #  'lens0_e1': float(tp["lens0_e1"]),#float(tp["lens0_e1"]), 
         #'lens0_e2': float(tp['lens0_e2']),
         'lens0_e1': dist.Uniform(tp['lens0_e1']-0.5, tp['lens0_e1']+0.5),#dist.Normal(tp['lens0_e1'], 0.01),
         'lens0_e2': dist.Uniform(tp['lens0_e2']-0.5, tp['lens0_e2']+0.5),#dist.Normal(tp['lens0_e2'], 0.01),
         'lens0_center_x': 0.0, #dist.Normal(0, lens_prior_sigma['lens0_center_x']),
         'lens0_center_y': 0.0, #dist.Normal(0, lens_prior_sigma['lens0_center_y']),
         #'lens1_gamma1': float(tp["lens1_gamma1"]),
         #'lens1_gamma2': float(tp["lens1_gamma2"]),
         'lens1_gamma1': dist.Uniform(tp['lens1_gamma1']-0.2, tp['lens1_gamma1']+0.2),#dist.Normal(tp['lens1_gamma1'], 0.01),
         'lens1_gamma2': dist.Uniform(tp['lens1_gamma2']-0.2, tp['lens1_gamma2']+0.2),#dist.Normal(tp['lens1_gamma2'], 0.01),
         'lens1_ra_0': 0.0,
         'lens1_dec_0': 0.0,
         'light0_R_sersic': float(tp["light0_R_sersic"]),
         'light0_n_sersic': float(tp["light0_n_sersic"]),
         'light0_amp': float(tp['light0_amp']),
         'light0_e1': float(tp["light0_e1"]),
         'light0_e2': float(tp["light0_e2"]),
         'light0_center_x': float(tp["light0_center_x"]),
         'light0_center_y': float(tp["light0_center_y"]),
        #   "noise_sigma_bkg": tp["noise_sigma_bkg"],
          'source0_n_sersic': dist.Uniform(tp['source0_n_sersic']-0.5, tp['source0_n_sersic']+0.5),#float(tp['source0_n_sersic']),
        #   'source0_e1': float(tp["source0_e1"]),
        #   'source0_e2': float(tp["source0_e2"]),
        #   'source0_center_x': float(tp["source0_center_x"]),
        #   'source0_center_y': float(tp["source0_center_y"]),
         #'y0gw': dist.Normal(float(tp['y0gw']), f*float(tp['y0gw'])),
         #'y1gw': dist.Normal(float(tp['y1gw']), *float(tp['y1gw']))
        } 
# cfg_gw_source = ctx_gw_source["cfg"]

In [ ]:
# import jax.numpy as jnp
# import numpy as np
# from gwemfish.data_sim import compute_gw_from_images

# run_inference(ctx_gw_source_both, mode="EM+GW", method="fisher-source",
#               cfg={"output": {"json_tag": "emgw_fisher_source_preflight"}})

# pm  = ctx_gw_source_both["likelihood"]["probmodel"]
# tp  = ctx_gw_source_both["truth_params"]
# kl  = ctx_gw_source_both["kwargs_lens"]
# src = ctx_gw_source_both["cfg"]["gw"]["source_pos"]

# thetas, _ = pm.solver.solve(jnp.array([float(src[0]), float(src[1])]), kl, **pm.solver_params)
# x_m, y_m  = np.array(thetas[:, 0]), np.array(thetas[:, 1])

# print(f"solver_params : {pm.solver_params}")
# print(f"\nModel images (helens, nsolutions={pm.solver_params['nsolutions']}):")
# for x, y in zip(x_m, y_m):
#     print(f"  x={x:.6f}  y={y:.6f}  r={np.hypot(x, y):.6f}")
# print("Mock images:")
# for x, y in zip(ctx_gw_source_both["x_img_gw"], ctx_gw_source_both["y_img_gw"]):
#     print(f"  x={float(x):.6f}  y={float(y):.6f}  r={np.hypot(float(x), float(y)):.6f}")

# r_m    = np.sort(np.hypot(x_m, y_m))
# r_mock = np.sort([np.hypot(float(x), float(y))
#                   for x, y in zip(ctx_gw_source_both["x_img_gw"], ctx_gw_source_both["y_img_gw"])])
# print(f"\nMax |Δr|  = {np.max(np.abs(r_m - r_mock)) if len(r_m)==len(r_mock) else float('nan'):.2e}  (should be < 1e-6)")

# _, td_m, *_ = compute_gw_from_images(jnp.array(x_m), jnp.array(y_m), kl, pm.lens_gw,
#                                       float(tp["T_star"]), float(tp["dL"]))
# td_m   = np.sort(np.array(td_m))
# td_obs = np.sort(np.array(ctx_gw_source_both["gw_obs"]["time_delays"]))
# print(f"Max |Δtd| = {np.max(np.abs(td_m - td_obs)):.2e} s  (should be < 0.01 s)")

In [ ]:
# p = ctx_gw_source_both["cfg"]["priors"]
# print(type(p.get("lens1_ra_0")), p.get("lens1_ra_0"))
# print(type(p.get("T_star")), p.get("T_star"))

In [ ]:
samples_emgw_deriv_approx_source, truths_emgw_deriv_approx_source = run_inference(
    ctx_gw_source_both,
    mode="EM+GW",
    method="deriv-approx-source",
    cfg={
        "output": {"json_tag": "emgw_deriv_approx_source"},
        "inference": {
            "informed": True,
            "regularize": False,
            "num_chains": 5,
            "num_warmup": 8000,
            "num_samples": 8000,
        },
    },
)

In [ ]:
src = ctx_gw_source_both["cfg"]["gw"]["source_pos"]
truths_emgw_plot = {
    **truths_emgw_deriv_approx_source,
    "y0gw": float(src[0]),
    "y1gw": float(src[1]),
}

_ = plot_posterior(
    samples_emgw_deriv_approx_source,
    truths_emgw_plot,
    cfg={
        "output": {"output_dir": None},
        "plot": {
            "plot_mode": "groupwise",
            "save_path": None,
            "quantiles": [0.16, 0.5, 0.84],
            "show_titles": True,
        },
    },
)

In [ ]:
from gwemfish.corner_plot_utils import create_default_param_groups, plot_multi_comparison_corner

keys_em   = set(samples_em_deriv_approx)
keys_emgw = set(samples_emgw_deriv_approx_source)
shared    = keys_em & keys_emgw

param_groups = {
    name: [k for k in keys if k in shared]
    for name, keys in create_default_param_groups(samples_emgw_deriv_approx_source).items()
}
param_groups = {name: keys for name, keys in param_groups.items() if keys}

src = ctx_gw_source_both["cfg"]["gw"]["source_pos"]
flat_truths = {
    **truths_emgw_deriv_approx_source,
    "y0gw": float(src[0]),
    "y1gw": float(src[1]),
}
truths_dict_nested = {
    name: {k: flat_truths[k] for k in keys if k in flat_truths}
    for name, keys in param_groups.items()
}

NORMALIZE = True

plot_multi_comparison_corner(
    [samples_em_deriv_approx, samples_emgw_deriv_approx_source],
    param_groups,
    labels=["EM-only deriv-approx", "EM+GW deriv-approx-source"],
    colors=["steelblue", "darkorange"],
    truths_dict=truths_dict_nested,
    save_path=None,
    hist_kwargs={"density": NORMALIZE},
)

In [ ]:
from gwemfish.corner_plot_utils import plot_multi_comparison_corner

COMPARE_PARAMS = ["T_star", "dL", "lens0_e2", "lens0_gamma", "y0gw", "y1gw"]
NORMALIZE = True

shared = set(samples_deriv_approx_source) & set(samples_emgw_deriv_approx_source)
plot_keys = [k for k in COMPARE_PARAMS if k in shared]
missing = [k for k in COMPARE_PARAMS if k not in shared]
if missing:
    print("skip (not free in both runs):", missing)
print("plotting:", plot_keys)

src = ctx_gw_source_both["cfg"]["gw"]["source_pos"]
flat_truths = {
    **truths_deriv_approx_source,
    **truths_emgw_deriv_approx_source,
    "y0gw": float(src[0]),
    "y1gw": float(src[1]),
}
param_groups = {"gw_science": plot_keys}
truths_dict_nested = {
    "gw_science": {k: flat_truths[k] for k in plot_keys if k in flat_truths},
}

plot_multi_comparison_corner(
    [samples_deriv_approx_source, samples_emgw_deriv_approx_source],
    param_groups,
    labels=["GW-only deriv-approx-source", "EM+GW deriv-approx-source"],
    colors=["steelblue", "darkorange"],
    truths_dict=truths_dict_nested,
    save_path=None,
    hist_kwargs={"density": NORMALIZE},
)
